This notebook validates the full data preparation and feature engineering pipeline.
It loads the cleaned dataset, applies transformations, and confirms the data is ready for modeling.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import pandas as pd
import numpy as np

from src.data622.dataset import (
    load_salary_data,
    filter_model_population,
    add_tenure_proxy,
    split_by_year
)

from src.data622.features import (
    add_feature_columns,
    make_preprocessor
)

In [2]:
# Load raw salary dataset and check initial shape
df = load_salary_data()
print("Raw shape:", df.shape)

df.head()

Raw shape: (2770022, 20)


,fiscal_year,payroll_number,agency_name,last_name,first_name,agency_start_date,work_location_borough,title_description,leave_status_as_of_june_30,base_salary,pay_basis,regular_hours,regular_gross_paid,ot_hours,total_ot_paid,total_other_pay,employee_agency_tenure,agency_std,title_category,title_std
0,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,BEREZIN,MIKHAIL,2015-08-10,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,4.889802,Office Of Emergency Management,Management,Emergency Preparedness Manager
1,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,GEAGER,VERONICA,2016-09-12,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,3.797399,Office Of Emergency Management,Management,Emergency Preparedness Manager
2,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,RAMANI,SHRADDHA,2016-02-22,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,4.353183,Office Of Emergency Management,Management,Emergency Preparedness Manager
3,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,ROTTA,JONATHAN,2013-09-16,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,6.787132,Office Of Emergency Management,Management,Emergency Preparedness Manager
4,2020,17.0,OFFICE OF EMERGENCY MANAGEMENT,WILSON II,ROBERT,2018-04-30,BROOKLYN,EMERGENCY PREPAREDNESS MANAGER,ACTIVE,86005.0,per Annum,1820.0,84698.21,0.0,0.0,0.0,2.168378,Office Of Emergency Management,Management,Emergency Preparedness Manager


In [3]:
# Filter dataset to model-ready population (valid salaries, key columns)
df = filter_model_population(df)

print("After cleaning:", df.shape)

df.head()

After cleaning: (629147, 7)


,fiscal_year,agency_std,title_std,title_category,base_salary,regular_hours,work_location_borough
0,2020,office of emergency management,emergency preparedness manager,management,86005.0,1820.0,brooklyn
1,2020,office of emergency management,emergency preparedness manager,management,94415.0,1820.0,brooklyn
2,2020,office of emergency management,emergency preparedness specialist,technical,67676.0,1820.0,brooklyn
3,2020,office of emergency management,emergency preparedness manager,management,83791.0,1820.0,brooklyn
4,2020,office of emergency management,emergency preparedness specialist,technical,73403.0,1820.0,brooklyn


In [4]:
# Re-run full pipeline up to feature engineering
df = load_salary_data()
df = filter_model_population(df)

# Add tenure proxy + engineered features
df = add_tenure_proxy(df)
df = add_feature_columns(df)

In [5]:
# Inspect newly created features
df[[
    "title_std",
    "title_frequency",
    "agency_size",
    "title_avg_salary"
]].head()

,title_std,title_frequency,agency_size,title_avg_salary
0,emergency preparedness manager,333,1209,109542.180180
1,emergency preparedness manager,333,1209,109542.180180
2,emergency preparedness specialist,691,1209,71478.479016
3,emergency preparedness manager,333,1209,109542.180180
4,emergency preparedness specialist,691,1209,71478.479016


In [6]:
# Summary statistics for base salary (before transformation)
df["base_salary"].describe()

count    629147.000000
mean      74440.897074
std       34355.497909
min       12052.820000
25%       49116.000000
50%       66413.000000
75%       89763.000000
max      414707.000000
Name: base_salary, dtype: float64

In [7]:
# Compare original vs log-transformed salary
df[["base_salary", "log_base_salary"]].head()

,base_salary,log_base_salary
0,86005.0,11.362161
1,94415.0,11.455455
2,67676.0,11.122487
3,83791.0,11.336081
4,73403.0,11.203720


In [8]:
# Split data into train, validation, and test sets by year
train_df, valid_df, test_df = split_by_year(df)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test:", test_df.shape)

Train: (511549, 14)
Valid: (58446, 14)
Test: (59152, 14)


In [9]:
# Import preprocessing utilities
from src.data622.features import add_feature_columns, make_preprocessor, get_model_columns

# Create preprocessing pipeline using training data
preprocessor = make_preprocessor(train_df)

# Get feature columns (categorical + numeric)
cat_cols, num_cols = get_model_columns(train_df)
feature_cols = cat_cols + num_cols

# Select feature matrix
X_train = train_df[feature_cols]

# Fit and transform features
Xt = preprocessor.fit_transform(X_train)

print("Transformed shape:", Xt.shape)

Transformed shape: (511549, 693)


In [10]:
# Quick sanity check
df = load_salary_data()
print(df.shape)
print(df.columns.tolist())

(2770022, 20)
['fiscal_year', 'payroll_number', 'agency_name', 'last_name', 'first_name', 'agency_start_date', 'work_location_borough', 'title_description', 'leave_status_as_of_june_30', 'base_salary', 'pay_basis', 'regular_hours', 'regular_gross_paid', 'ot_hours', 'total_ot_paid', 'total_other_pay', 'employee_agency_tenure', 'agency_std', 'title_category', 'title_std']
